In [1]:
import duckdb as ddb
import pandas as pd

# l. Exploración incial
¿Cuántos registos? ¿Qué periodo cubren?

In [2]:
# Conectar a ddb en memoria
con = ddb.connect()

# Leer JSON
con.execute("""
CREATE TABLE logs AS
SELECT * FROM read_json_auto('../data/*.json')
""")

# Ver estructura
con.sql("""
DESCRIBE logs;
""")

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ log_id           │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ timestamp        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ service_id       │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ server_id        │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ trace_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ span_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ method           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ endpoint         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ status_code      │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ response_time_ms │ JSON

In [3]:
con.sql("""
    SELECT 
        endpoint,
        COUNT(*) as hits,
    FROM logs
    GROUP BY endpoint
""").show()

┌─────────────────────┬───────┐
│      endpoint       │ hits  │
│       varchar       │ int64 │
├─────────────────────┼───────┤
│ /api/checkout       │   440 │
│ /api/cart           │   392 │
│                     │   227 │
│ NULL                │  1517 │
│ /METRICS            │     8 │
│ /api/products\n     │     2 │
│   /api/users        │     6 │
│ /api/auth/login\n   │     2 │
│ /api/cart\n         │     4 │
│ /api/auth/logout    │   403 │
│       ·             │    ·  │
│       ·             │    ·  │
│       ·             │    ·  │
│ /api/payments       │   435 │
│ /API/SEARCH         │     3 │
│   /api/payments     │     9 │
│   /api/auth/login   │     9 │
│   /metrics          │     5 │
│   /api/cart         │     5 │
│ /API/PRODUCTS       │     8 │
│ /metrics\n          │     9 │
│ /api/checkout\n     │     3 │
│ /api/orders\n       │     7 │
└─────────────────────┴───────┘
  46 rows (20 shown)2 columns



In [4]:
# Normalizar una vista saneada, remplazo nulls y celdas vacias con un valor fijo y casteo tipos de datos, corrijo strings con mayusculas
con.execute("DROP VIEW IF EXISTS v_logs;")
con.execute("""
CREATE VIEW v_logs AS 
SELECT
    * EXCLUDE (endpoint, status_code, response_time_ms, timestamp),
    COALESCE(
        NULLIF(
            TRIM(
                LOWER(
                    REGEXP_REPLACE(endpoint, '[\r\n\t]', '', 'g')
                )
            ), 
            ''
        ), 
        'unknown_endpoint'
    ) AS endpoint,
    COALESCE(NULLIF(TRIM(endpoint), ''), 'UNKNOWN_ENDPOINT') AS endpoint,
    TRY_CAST(status_code AS INTEGER) AS status_code,
    TRY_CAST(response_time_ms AS DOUBLE) AS response_time_ms,
    CAST(
        CASE 
            WHEN LENGTH(TRIM(CAST(timestamp AS VARCHAR))) = 0 THEN NULL
            ELSE COALESCE(
                TRY_CAST(timestamp AS TIMESTAMP),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%m/%d/%Y'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%d %b %Y'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%Y/%m/%d'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%d-%m-%Y'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%b %d, %Y')
            )
        END AS TIMESTAMP
    ) AS timestamp
FROM logs;
""")
con.sql("""
    SELECT 
        endpoint,
        COUNT(*) as hits,
    FROM v_logs
    GROUP BY endpoint
""").show()

┌──────────────────┬───────┐
│     endpoint     │ hits  │
│     varchar      │ int64 │
├──────────────────┼───────┤
│ /api/auth/logout │   416 │
│ /health          │   423 │
│ /api/search      │   425 │
│ /api/orders      │   428 │
│ /api/auth/login  │   406 │
│ /api/users       │   420 │
│ unknown_endpoint │  1744 │
│ /api/payments    │   457 │
│ /api/checkout    │   452 │
│ /api/cart        │   403 │
│ /api/products    │   442 │
│ /metrics         │   452 │
└──────────────────┴───────┘
  12 rows        2 columns



In [5]:
con.sql("""
SELECT
    COUNT(*) as total_requests,
    MIN(timestamp) as primera_request,
    MAX(timestamp) as ultima_request,
    COUNT(DISTINCT user_id) as usuarios_unicos,
    COUNT(DISTINCT endpoint) as endpoints_unicos
FROM v_logs
WHERE timestamp IS NOT NULL;
""").show()


┌────────────────┬─────────────────────┬─────────────────────┬─────────────────┬──────────────────┐
│ total_requests │   primera_request   │   ultima_request    │ usuarios_unicos │ endpoints_unicos │
│     int64      │      timestamp      │      timestamp      │      int64      │      int64       │
├────────────────┼─────────────────────┼─────────────────────┼─────────────────┼──────────────────┤
│           5690 │ 2024-01-01 04:28:47 │ 2027-08-14 00:00:00 │            2495 │               12 │
└────────────────┴─────────────────────┴─────────────────────┴─────────────────┴──────────────────┘



# ll. Análisis de tráfico

Qué endpoints reciben más tráfico

In [6]:
con.sql("""
    SELECT 
        endpoint,
        COUNT(*) as hits,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM logs), 2) as porcentaje
    FROM v_logs
    GROUP BY endpoint
    ORDER BY hits DESC
    LIMIT 10;
""").show()

┌──────────────────┬───────┬────────────┐
│     endpoint     │ hits  │ porcentaje │
│     varchar      │ int64 │   double   │
├──────────────────┼───────┼────────────┤
│ unknown_endpoint │  1744 │      26.96 │
│ /api/payments    │   457 │       7.07 │
│ /metrics         │   452 │       6.99 │
│ /api/checkout    │   452 │       6.99 │
│ /api/products    │   442 │       6.83 │
│ /api/orders      │   428 │       6.62 │
│ /api/search      │   425 │       6.57 │
│ /health          │   423 │       6.54 │
│ /api/users       │   420 │       6.49 │
│ /api/auth/logout │   416 │       6.43 │
└──────────────────┴───────┴────────────┘
  10 rows                     3 columns



# lll. Análisis de errores
Los errores 5xx son críticos. Afectan la experiencia del usuario y pueden indicar bugs

In [7]:
con.sql("""
SELECT
    status_code,
    COUNT(*) as status_count
FROM v_logs
GROUP BY status_code
ORDER BY status_count DESC;
""").show()

┌─────────────┬──────────────┐
│ status_code │ status_count │
│    int32    │    int64     │
├─────────────┼──────────────┤
│        NULL │         1741 │
│         200 │         1283 │
│         404 │          362 │
│         401 │          360 │
│         400 │          348 │
│         201 │          346 │
│         204 │          344 │
│         403 │          342 │
│         500 │          341 │
│         301 │          339 │
│         502 │          335 │
│         503 │          327 │
└─────────────┴──────────────┘
  12 rows          2 columns



In [8]:
con.sql("""
SELECT
    endpoint,
    COUNT(*) as total_errors,
    COUNT(DISTINCT user_id) as usuarios_afectados,
    ROUND(AVG(TRY_CAST(response_time_ms AS DOUBLE)), 2) as avg_response_time
FROM v_logs
WHERE status_code >= 500
GROUP BY endpoint
ORDER BY total_errors DESC
LIMIT 10
""").show()

┌──────────────────┬──────────────┬────────────────────┬───────────────────┐
│     endpoint     │ total_errors │ usuarios_afectados │ avg_response_time │
│     varchar      │    int64     │       int64        │      double       │
├──────────────────┼──────────────┼────────────────────┼───────────────────┤
│ /metrics         │           98 │                 65 │          84844.79 │
│ /api/orders      │           94 │                 58 │          16086.92 │
│ /api/products    │           92 │                 57 │          83599.09 │
│ /api/users       │           86 │                 51 │           15281.0 │
│ /api/checkout    │           86 │                 59 │          14745.61 │
│ /api/payments    │           85 │                 49 │          14716.54 │
│ /api/search      │           83 │                 48 │          16932.12 │
│ /api/auth/login  │           82 │                 47 │          14141.78 │
│ /api/auth/logout │           81 │                 48 │          16087.36 │

# lV. Análisis de performance por endpoint
¿Qué endpoints son más lentos? El promedio puede ser engañoso. El percentil 95 (p95) me dice cuánto tarda el 95% de las requests.

In [18]:
con.sql("""
SELECT 
    endpoint,
    COUNT(*) AS requests,
    ROUND(AVG(response_time_ms), 2) AS avg_time,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY response_time_ms), 2) AS p50,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY response_time_ms), 2) AS p95,
    ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY response_time_ms), 2) AS p99,
    ROUND(MAX(response_time_ms), 2) AS max_response_time
FROM v_logs
WHERE response_time_ms IS NOT NULL AND status_code < 500
GROUP BY endpoint
HAVING COUNT(*) > 100
ORDER BY p95 DESC;
""").show()

┌──────────────────┬──────────┬──────────┬────────┬────────┬──────────┬───────────────────┐
│     endpoint     │ requests │ avg_time │  p50   │  p95   │   p99    │ max_response_time │
│     varchar      │  int64   │  double  │ double │ double │  double  │      double       │
├──────────────────┼──────────┼──────────┼────────┼────────┼──────────┼───────────────────┤
│ /api/cart        │      281 │  2052.57 │  242.0 │  490.0 │    500.0 │          433636.0 │
│ /api/auth/logout │      274 │  1131.09 │  237.5 │  483.7 │    500.0 │          125392.0 │
│ /api/orders      │      273 │  1785.52 │  230.0 │  483.0 │   499.28 │          225803.0 │
│ unknown_endpoint │      290 │  2311.17 │  251.5 │ 482.55 │ 60480.86 │          276338.0 │
│ /api/payments    │      306 │  2011.14 │  254.5 │ 481.25 │  10209.0 │          255807.0 │
│ /api/auth/login  │      256 │   1394.6 │  256.0 │ 478.25 │   495.45 │          166690.0 │
│ /api/checkout    │      293 │  1482.69 │  284.0 │  478.2 │ 15157.92 │         

In [10]:
con.sql("""
        WITH error_5xx AS (
            SELECT 
                endpoint,
                COUNT(*) as error_count
            FROM v_logs
            WHERE status_code >= 500
            GROUP BY endpoint
        ),
        total_requests AS (
            SELECT 
                endpoint,
                COUNT(*) as total_count
            FROM v_logs
            GROUP BY endpoint
        ),
        avg_time_response_ms AS (
            SELECT
                endpoint,
                ROUND(AVG(response_time_ms), 2) as avg_response_time_ms
            FROM v_logs
            GROUP BY endpoint
        )
        SELECT 
            error_5xx.endpoint,
            ROUND(error_5xx.error_count * 100.0 / total_requests.total_count, 2) AS error_percentage,
            error_5xx.error_count,
            total_requests.total_count,
            avg_time_response_ms.avg_response_time_ms
        FROM error_5xx
        JOIN total_requests USING (endpoint)
        JOIN avg_time_response_ms USING (endpoint)
        ORDER BY error_percentage DESC
        """).show()

┌──────────────────┬──────────────────┬─────────────┬─────────────┬──────────────────────┐
│     endpoint     │ error_percentage │ error_count │ total_count │ avg_response_time_ms │
│     varchar      │      double      │    int64    │    int64    │        double        │
├──────────────────┼──────────────────┼─────────────┼─────────────┼──────────────────────┤
│ /api/orders      │            21.96 │          94 │         428 │              5165.26 │
│ /metrics         │            21.68 │          98 │         452 │             18871.36 │
│ /api/products    │            20.81 │          92 │         442 │             16904.96 │
│ /api/users       │            20.48 │          86 │         420 │              5000.95 │
│ /api/auth/login  │             20.2 │          82 │         406 │             18729.01 │
│ /api/search      │            19.53 │          83 │         425 │              4206.76 │
│ /api/auth/logout │            19.47 │          81 │         416 │              4165.88 │

# V. Tendencia horaria
¿A qué hora hay más tráfico? El tráfico varía por hora, saber cuándo hay picos ayuda a planificar capacidad

In [11]:
con.sql("""
SELECT
    EXTRACT(HOUR FROM timestamp) as hora,
    COUNT(*) as requests,
    ROUND(AVG(response_time_ms), 2) as avg_response_time,
    SUM(CASE WHEN status_code >= 500 THEN 1 ELSE 0 END) as errors
FROM v_logs
GROUP BY EXTRACT(HOUR FROM timestamp)
ORDER BY hora
""").show()

┌───────┬──────────┬───────────────────┬────────┐
│ hora  │ requests │ avg_response_time │ errors │
│ int64 │  int64   │      double       │ int128 │
├───────┼──────────┼───────────────────┼────────┤
│     0 │      593 │           5455.08 │     92 │
│     1 │      213 │           4572.25 │     42 │
│     2 │      201 │           7347.51 │     37 │
│     3 │      213 │          14587.41 │     31 │
│     4 │      228 │           4142.28 │     38 │
│     5 │      230 │           3127.88 │     31 │
│     6 │      247 │           3407.81 │     44 │
│     7 │      232 │           6316.06 │     42 │
│     8 │      209 │            3491.4 │     34 │
│     9 │      226 │           4411.01 │     44 │
│     · │       ·  │              ·    │      · │
│     · │       ·  │              ·    │      · │
│     · │       ·  │              ·    │      · │
│    15 │      201 │           3666.95 │     31 │
│    16 │      214 │           3196.44 │     33 │
│    17 │      235 │           3167.36 │     33 │


Top 3 requests más lentos por endpoint utilizando window function

In [12]:
con.sql("""
WITH ranked AS (
    SELECT
        endpoint,
        timestamp,
        response_time_ms,
        user_id,
        ROW_NUMBER() OVER (
            PARTITION BY endpoint
            ORDER BY response_time_ms DESC
        ) as rank
    FROM v_logs
    WHERE status_code < 500
)
SELECT * FROM ranked
WHERE rank <= 3
ORDER BY endpoint, rank;
""").show()

┌──────────────────┬─────────────────────┬──────────────────┬─────────┬───────┐
│     endpoint     │      timestamp      │ response_time_ms │ user_id │ rank  │
│     varchar      │      timestamp      │      double      │  json   │ int64 │
├──────────────────┼─────────────────────┼──────────────────┼─────────┼───────┤
│ /api/auth/login  │ 2026-02-13 10:20:32 │         166690.0 │ 4064    │     1 │
│ /api/auth/login  │ 2025-08-01 00:37:49 │         127155.0 │ 301     │     2 │
│ /api/auth/login  │ 2025-11-29 22:07:16 │            496.0 │ 8703    │     3 │
│ /api/auth/logout │ 2026-08-08 00:03:14 │         125392.0 │ 103316  │     1 │
│ /api/auth/logout │ 2026-04-19 10:02:22 │         120696.0 │ 387     │     2 │
│ /api/auth/logout │ 2025-09-02 00:37:34 │            500.0 │ 8146    │     3 │
│ /api/cart        │ 2027-08-11 00:00:00 │         433636.0 │ 6157    │     1 │
│ /api/cart        │ 2026-07-04 04:40:31 │          73920.0 │ 2559    │     2 │
│ /api/cart        │ 2026-03-28 02:39:15

Comparación con periodo anterior ¿Cómo cambia el tráfico día a día?

In [13]:
df_res = con.sql("""
WITH daily_stats AS (
    SELECT 
        DATE(timestamp) as fecha,
        COUNT(*) as requests,
        ROUND(AVG(response_time_ms), 2) as avg_time
    FROM v_logs
    GROUP BY DATE(timestamp)
)
SELECT
    fecha,
    requests,
    LAG(requests) OVER (ORDER BY fecha) as requests_dia_anterior,
    requests - LAG(requests) OVER (ORDER BY fecha) as diferencia,
    ROUND(
        (requests - LAG(requests) OVER (ORDER BY fecha)) * 100.0 /
        LAG(requests) OVER (ORDER BY fecha), 2
    ) as cambio_porcentual
FROM daily_stats
WHERE fecha IS NOT NULL
ORDER BY fecha;
"""
).df()

df_res

,fecha,requests,requests_dia_anterior,diferencia,cambio_porcentual
0,2024-01-01,8,<NA>,<NA>,NaN
1,2024-01-02,3,8,-5,-62.50
2,2024-01-03,5,3,2,66.67
3,2024-01-04,7,5,2,40.00
4,2024-01-05,8,7,1,14.29
...,...,...,...,...,...
1043,2027-08-05,1,1,0,0.00
1044,2027-08-06,1,1,0,0.00
1045,2027-08-07,1,1,0,0.00
1046,2027-08-11,1,1,0,0.00


In [14]:
con.sql("""
        WITH daily_stats AS (
            SELECT 
                DATE(timestamp) as date,
                COUNT(*) as requests,
                ROUND(AVG(response_time_ms), 2) as avg_response_time
            FROM v_logs
            GROUP BY DATE(timestamp)
        ), 
        daily_percent_change AS (
        SELECT 
            date,
            requests,
            requests - LAG(requests) OVER (ORDER BY date) as difference,
            ROUND(
                (requests - LAG(requests) OVER (ORDER BY date)) * 100.0 / 
                LAG(requests) OVER (ORDER BY date), 
                2
            ) as percent_change
        FROM daily_stats
        )
        
        SELECT AVG(percent_change) AS avg_daily_percent_change FROM daily_percent_change;
        """).show()

┌──────────────────────────┐
│ avg_daily_percent_change │
│          double          │
├──────────────────────────┤
│       103.77242366412214 │
└──────────────────────────┘



In [15]:
con.sql("""
WITH requests_per_hour AS (
    SELECT 
        EXTRACT(HOUR FROM timestamp) AS hour,
        COUNT(*) AS request_count
    FROM v_logs
    WHERE timestamp IS NOT NULL
    GROUP BY EXTRACT(HOUR FROM timestamp)
)
SELECT 
    EXTRACT(HOUR FROM timestamp) AS peak_hour,
    COUNT(*) AS request_count
FROM v_logs
WHERE timestamp IS NOT NULL
GROUP BY EXTRACT(HOUR FROM timestamp)
HAVING COUNT(*) > (SELECT AVG(request_count) FROM requests_per_hour)
ORDER BY request_count DESC;
""").show()

┌───────────┬───────────────┐
│ peak_hour │ request_count │
│   int64   │     int64     │
├───────────┼───────────────┤
│         0 │           593 │
│        18 │           251 │
│         6 │           247 │
└───────────┴───────────────┘



La hora 0 esta inflada debido a registros sin hora que el casteo puso por default en hora 00:00:00

In [16]:
con.sql("""
WITH requests_per_hour AS (
    SELECT 
        EXTRACT(HOUR FROM timestamp) AS hour,
        COUNT(*) AS request_count
    FROM v_logs
    WHERE timestamp IS NOT NULL 
      AND EXTRACT(HOUR FROM timestamp) != 0  -- Excluimos la hora 0 acumulada por defecto
    GROUP BY EXTRACT(HOUR FROM timestamp)
)
SELECT 
    EXTRACT(HOUR FROM timestamp) AS peak_hour,
    COUNT(*) AS request_count
FROM v_logs
WHERE timestamp IS NOT NULL 
  AND EXTRACT(HOUR FROM timestamp) != 0
GROUP BY EXTRACT(HOUR FROM timestamp)
HAVING COUNT(*) > (SELECT AVG(request_count) FROM requests_per_hour)
ORDER BY request_count DESC;
""").show()

┌───────────┬───────────────┐
│ peak_hour │ request_count │
│   int64   │     int64     │
├───────────┼───────────────┤
│        18 │           251 │
│         6 │           247 │
│        14 │           236 │
│        17 │           235 │
│        10 │           234 │
│         7 │           232 │
│         5 │           230 │
│         4 │           228 │
│        23 │           228 │
│        11 │           226 │
│         9 │           226 │
│        20 │           223 │
└───────────┴───────────────┘
  12 rows         2 columns



# Vl. Análisis de errores según método HTTP


In [17]:
con.sql("""
WITH errors_by_method AS (
    SELECT 
        UPPER(TRIM(method)) AS method,
        endpoint,
        COUNT(*) AS error_5xx_count,
        ROUND(AVG(response_time_ms), 2) AS avg_response_time_ms
    FROM v_logs
    WHERE status_code >= 500 
      AND method IS NOT NULL 
      AND LENGTH(TRIM(CAST(method AS VARCHAR))) > 0
    GROUP BY UPPER(TRIM(method)), endpoint
), 
ranked_errors AS (
    SELECT 
        method, 
        endpoint, 
        error_5xx_count,
        avg_response_time_ms,
        DENSE_RANK() OVER (PARTITION BY method ORDER BY error_5xx_count DESC) AS error_rank
    FROM errors_by_method
)
SELECT * 
FROM ranked_errors 
WHERE error_rank <= 3 
  AND error_5xx_count > 3
ORDER BY method, error_rank;
""").show()

┌─────────┬──────────────────┬─────────────────┬──────────────────────┬────────────┐
│ method  │     endpoint     │ error_5xx_count │ avg_response_time_ms │ error_rank │
│ varchar │     varchar      │      int64      │        double        │   int64    │
├─────────┼──────────────────┼─────────────────┼──────────────────────┼────────────┤
│ DELETE  │ /api/products    │              13 │             13084.77 │          1 │
│ DELETE  │ /api/auth/logout │              13 │             17496.62 │          1 │
│ DELETE  │ /api/payments    │              12 │             11899.83 │          2 │
│ DELETE  │ /api/checkout    │              11 │              14360.8 │          3 │
│ DELETE  │ /api/search      │              11 │              16671.1 │          3 │
│ GET     │ /metrics         │              38 │             15326.28 │          1 │
│ GET     │ /api/orders      │              38 │             16899.16 │          1 │
│ GET     │ /api/products    │              37 │            17239